In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

from src.utils import load_full_config
from src.model import SimpleTransformer


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os,sys
sys.path.append('../src')
from utils import load_full_config
from model import SimpleTransformer

cfg = load_full_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load data
data_path = os.path.join('..', cfg['paths']['mgf_data_path'])
data = torch.load(data_path, weights_only=False)

trajectories = data['trajectories']
targets = data['targets']
s_range = data['s_range']

# Load model
model = SimpleTransformer(**cfg['architecture'])
model_path = os.path.join(
    '..',
    cfg['paths']['save_dir'],
    cfg['paths'].get('mgf_model_name', 'model_mgf.pth')
)

if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    # Handle both old format (direct state dict) and new format (checkpoint dict)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        # New format with loss history
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        # Old format (direct state dict)
        model.load_state_dict(checkpoint)
        
    model.to(device)
    model.eval()
    print(f"Loaded model from {model_path}")
else:
    raise FileNotFoundError(
        f"Model not found at {model_path}. Train first with python ../scripts/train.py"
    )

In [ ]:
idx = np.random.randint(0, len(trajectories))
traj = trajectories[idx:idx+1].to(device)
true_phi = targets[idx].numpy()

with torch.no_grad():
    preds, maps = model(traj)
    pred_phi = preds[0, -1, :].cpu().numpy()

attn = maps[-1][0].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(traj[0].cpu().squeeze().numpy())
axes[0].set_title(f"Trajectory {idx}")
axes[0].set_xlabel("Time Step")
axes[0].set_ylabel("X(t)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(s_range, true_phi, label='True', color='black')
axes[1].plot(s_range, pred_phi, label='Predicted', color='red', linestyle='--')
axes[1].set_title("Log MGF Prediction")
axes[1].set_xlabel("s")
axes[1].set_ylabel("phi(s)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

sns.heatmap(attn, ax=axes[2], cmap='viridis')
axes[2].set_title("Attention Map (Top Layer)")
axes[2].set_xlabel("Key Position")
axes[2].set_ylabel("Query Position")

plt.tight_layout()
plt.show()
